# Regret minimization & CFR — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def expected_payoff(A,p,q): return float(np.asarray(p) @ np.asarray(A) @ np.asarray(q))
def best_response_payoffs(A,q): return np.asarray(A) @ np.asarray(q)
print('setup ok')

## Regret matching drives average play toward equilibrium
Regret minimization learns by asking how much better each action would have done in hindsight. These examples compute external regret, update regret-matching probabilities, show regret decay in matching pennies, isolate counterfactual regret at an information set, and visualize a tiny CFR-style averaging process.

In [ ]:
# 1) External regret compares the algorithm to the best fixed action
payoffs=np.array([[1,-1],[-1,1],[1,-1],[-1,1]])  # columns are actions H,T payoffs over four rounds
played=np.array([0,0,1,1]); alg=payoffs[np.arange(4),played].sum(); fixed=payoffs.sum(axis=0); regret=fixed.max()-alg
plt.figure(figsize=(5,3)); plt.bar(['algorithm','best fixed H','best fixed T'],[alg,fixed[0],fixed[1]]); plt.title('external regret = best fixed - actual')
assert alg==0 and fixed.max()==0 and regret==0

In [ ]:
# 2) Regret matching turns positive regrets into probabilities
reg=np.array([3.,1.]); strat=reg/reg.sum()
plt.figure(figsize=(5,3)); plt.bar(['H','T'],strat); plt.ylim(0,1); plt.title('positive regrets [3,1] -> probabilities [0.75,0.25]')
assert np.allclose(strat,[.75,.25])

In [ ]:
# 3) Regret matching in matching pennies approaches the 0.5 mix
A=np.array([[1,-1],[-1,1]]); regrets=np.zeros(2); avg=np.zeros(2); opp=np.array([.5,.5]); hist=[]
for t in range(1,501):
    pos=np.maximum(regrets,0); strat=pos/pos.sum() if pos.sum()>0 else np.ones(2)/2
    util=A@opp; regrets += util - strat@util; avg += strat; hist.append((avg/t).copy())
hist=np.array(hist)
plt.figure(figsize=(5,3)); plt.plot(hist[:,0],label='avg P(H)'); plt.axhline(.5,c='k',ls='--'); plt.legend(); plt.title('average strategy stays at Nash mix')
assert abs(hist[-1,0]-.5)<1e-12

In [ ]:
# 4) Counterfactual regret weights an information set by reach probability
reach=.25; action_values=np.array([2.,-1.]); node_value=.5*2+.5*(-1); cf_regret=reach*(action_values-node_value)
plt.figure(figsize=(5,3)); plt.bar(['action A','action B'],cf_regret); plt.axhline(0,c='k'); plt.title('counterfactual regret scales local advantage by reach')
assert np.allclose(cf_regret,[0.375,-0.375])

In [ ]:
# 5) CFR-style averaging: a dominated action probability shrinks
regrets=np.zeros(2); avg=np.zeros(2); hist=[]
for t in range(1,101):
    pos=np.maximum(regrets,0); strat=pos/pos.sum() if pos.sum()>0 else np.ones(2)/2
    values=np.array([1.,0.]); regrets += values - strat@values; avg += strat; hist.append((avg/t).copy())
hist=np.array(hist)
plt.figure(figsize=(5,3)); plt.plot(hist[:,0],label='avg good action'); plt.plot(hist[:,1],label='avg bad action'); plt.legend(); plt.title('positive regret concentrates play on better action')
assert hist[-1,0]>.9 and hist[-1,1]<.1